# Data Cleaning and Feature Engineering

In this notebook, we will clean the combined Citi Bike dataset
and create useful features for analysis.

In [1]:
import pandas as pd
import numpy as np

# Load raw combined dataset
combined_df = pd.read_csv(
    "../data/processed/citibike_raw_combined.csv",
    low_memory=False
)

print("Raw dataset loaded successfully!")
print("Shape:", combined_df.shape)

Raw dataset loaded successfully!
Shape: (7324003, 13)


## 1. Data Cleaning

### 1.1 Check Missing Values

In [2]:
combined_df.isnull().sum()

ride_id                   0
rideable_type             0
started_at                0
ended_at                  0
start_station_name     2290
start_station_id       2290
end_station_name      16498
end_station_id        17553
start_lat              2290
start_lng              2290
end_lat               17502
end_lng               17502
member_casual             0
dtype: int64

### 1.2 Check Duplicate Records

In [3]:
combined_df.duplicated().sum()

np.int64(0)

### 1.3 Check Data Types

In [4]:
combined_df.dtypes

ride_id                   str
rideable_type             str
started_at                str
ended_at                  str
start_station_name        str
start_station_id          str
end_station_name          str
end_station_id            str
start_lat             float64
start_lng             float64
end_lat               float64
end_lng               float64
member_casual             str
dtype: object

### 1.4 Convert Date and Time Columns

In [5]:
combined_df["started_at"] = pd.to_datetime(combined_df["started_at"])
combined_df["ended_at"] = pd.to_datetime(combined_df["ended_at"])

print("Date and time conversion completed!")

Date and time conversion completed!


### 1.5 Calculate Trip Duration

In [6]:
combined_df["trip_duration"] = (
    combined_df["ended_at"] - combined_df["started_at"]
).dt.total_seconds() / 60

print("Trip duration calculated successfully!")

Trip duration calculated successfully!


### 1.6 Check Trip Duration

In [7]:
combined_df["trip_duration"].describe()

count    7.324003e+06
mean     1.070612e+01
std      2.472693e+01
min      1.000300e+00
25%      4.558833e+00
50%      7.536883e+00
75%      1.265234e+01
max      1.559937e+03
Name: trip_duration, dtype: float64

### 1.7 Check Invalid Trip Durations

In [8]:
print("Zero or negative durations:", (combined_df["trip_duration"] <= 0).sum())
print("Durations over 24 hours:", (combined_df["trip_duration"] > 1440).sum())

Zero or negative durations: 0
Durations over 24 hours: 1152


### 1.8 Remove Invalid Trip Durations

In [9]:
combined_df = combined_df[
    (combined_df["trip_duration"] > 0) &
    (combined_df["trip_duration"] <= 1440)
]
print("Data after removing invalid durations:", combined_df.shape)

Data after removing invalid durations: (7322851, 14)


### 1.9 Handle Missing Station Information

In [10]:
station_columns = [
    "start_station_name",
    "start_station_id",
    "end_station_name",
    "end_station_id",
    "start_lat",
    "start_lng",
    "end_lat",
    "end_lng"
]

combined_df[station_columns].isnull().sum()

start_station_name     2290
start_station_id       2290
end_station_name      15375
end_station_id        16430
start_lat              2290
start_lng              2290
end_lat               16430
end_lng               16430
dtype: int64

In [11]:
combined_df = combined_df.dropna(
    subset=[
        "start_lat",
        "start_lng",
        "end_lat",
        "end_lng"
    ]
)

print("Data after removing missing coordinates:", combined_df.shape)

Data after removing missing coordinates: (7304514, 14)


### 1.10 Check Duplicate Ride IDs

In [12]:
combined_df["ride_id"].duplicated().sum()

np.int64(0)

### 1.11 Clean Station Names

In [13]:
station_name_columns = [
    "start_station_name",
    "end_station_name"
]

for column in station_name_columns:
    combined_df[column] = combined_df[column].str.strip()

print("Station names cleaned successfully")

Station names cleaned successfully


### 1.12 Check Coordinate Validity

In [14]:
print("Invalid start latitude:",
      ((combined_df["start_lat"] < -90) |
       (combined_df["start_lat"] > 90)).sum())

print("Invalid start longitude:",
      ((combined_df["start_lng"] < -180) |
       (combined_df["start_lng"] > 180)).sum())

print("Invalid end latitude:",
      ((combined_df["end_lat"] < -90) |
       (combined_df["end_lat"] > 90)).sum())

print("Invalid end longitude:",
      ((combined_df["end_lng"] < -180) |
       (combined_df["end_lng"] > 180)).sum())

Invalid start latitude: 0
Invalid start longitude: 0
Invalid end latitude: 0
Invalid end longitude: 0


### 1.13 Check Remaining Missing Values

In [15]:
combined_df.isnull().sum()

ride_id               0
rideable_type         0
started_at            0
ended_at              0
start_station_name    0
start_station_id      0
end_station_name      0
end_station_id        0
start_lat             0
start_lng             0
end_lat               0
end_lng               0
member_casual         0
trip_duration         0
dtype: int64

## 2. Feature Engineering

### 2.1 Extract Date and Time Features

In [16]:
combined_df["hour"] = combined_df["started_at"].dt.hour
combined_df["day_of_week"] = combined_df["started_at"].dt.day_name()
combined_df["month"] = combined_df["started_at"].dt.month
combined_df["is_weekend"] = combined_df["started_at"].dt.dayofweek >= 5

combined_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,trip_duration,hour,day_of_week,month,is_weekend
0,ACCC919B5A3CD9AD,electric_bike,2025-01-01 14:52:26.542,2025-01-01 14:59:53.427,W 20 St & 7 Ave,6182.02,E 10 St & 2 Ave,5746.02,40.742388,-73.997262,40.729708,-73.986598,casual,7.448083,14,Wednesday,1,False
1,1FABDF3EE40FCB0E,classic_bike,2025-01-10 05:03:13.646,2025-01-10 05:13:13.331,W 20 St & 7 Ave,6182.02,E 25 St & 1 Ave,6004.07,40.742388,-73.997262,40.738177,-73.977387,member,9.994750,5,Friday,1,False
2,88F0F3CFCBC79652,classic_bike,2025-01-13 13:40:17.630,2025-01-13 13:47:05.817,St James Pl & Oliver St,5238.05,Fulton St & William St,5137.11,40.713079,-73.998512,40.709601,-74.006551,member,6.803117,13,Monday,1,False
3,6FDE4E191D58E453,electric_bike,2025-01-10 08:29:16.996,2025-01-10 08:34:49.360,St James Pl & Oliver St,5238.05,Fulton St & William St,5137.11,40.713079,-73.998512,40.709601,-74.006551,member,5.539400,8,Friday,1,False
4,E9B03B9F77A85455,electric_bike,2025-01-11 18:59:48.427,2025-01-11 19:13:21.292,E 33 St & 1 Ave,6197.08,Allen St & Rivington St,5414.06,40.743227,-73.974498,40.720196,-73.989978,member,13.547750,18,Saturday,1,True


### 2.2 Extract Trip Date

In [17]:
combined_df["trip_date"] = combined_df["started_at"].dt.date

combined_df[["started_at", "trip_date"]].head()

,started_at,trip_date
0,2025-01-01 14:52:26.542,2025-01-01
1,2025-01-10 05:03:13.646,2025-01-10
2,2025-01-13 13:40:17.630,2025-01-13
3,2025-01-10 08:29:16.996,2025-01-10
4,2025-01-11 18:59:48.427,2025-01-11


### 2.3 Calculate Trip Distance

In [18]:
import numpy as np

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in kilometers

    lat1, lon1, lat2, lon2 = map(
        np.radians, [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c


combined_df["distance_km"] = haversine_distance(
    combined_df["start_lat"],
    combined_df["start_lng"],
    combined_df["end_lat"],
    combined_df["end_lng"]
)

combined_df[["start_lat", "end_lat", "distance_km"]].head()

,start_lat,end_lat,distance_km
0,40.742388,40.729708,1.671903
1,40.742388,40.738177,1.738770
2,40.713079,40.709601,0.780190
3,40.713079,40.709601,0.780190
4,40.743227,40.720196,2.873991


### 2.4 Calculate Average Speed

In [19]:
combined_df["speed_kmh"] = (
    combined_df["distance_km"] /
    (combined_df["trip_duration"] / 60)
)

combined_df[["distance_km", "trip_duration", "speed_kmh"]].head()

,distance_km,trip_duration,speed_kmh
0,1.671903,7.448083,13.468456
1,1.738770,9.994750,10.438098
2,0.780190,6.803117,6.880878
3,0.780190,5.539400,8.450630
4,2.873991,13.547750,12.728272


### 2.5 Check Invalid Speed Values

In [20]:
combined_df["speed_kmh"].describe()

count    7.304514e+06
mean     1.173570e+01
std      1.082812e+02
min      0.000000e+00
25%      9.039765e+00
50%      1.164342e+01
75%      1.439011e+01
max      1.468322e+05
Name: speed_kmh, dtype: float64

In [21]:
# Check unrealistic speed values

print("Zero speed:", (combined_df["speed_kmh"] == 0).sum())

print("Speed above 40 km/h:",
      (combined_df["speed_kmh"] > 40).sum())

print("Speed above 100 km/h:",
      (combined_df["speed_kmh"] > 100).sum())

Zero speed: 112851
Speed above 40 km/h: 20
Speed above 100 km/h: 14


### 2.6 Remove Unrealistic Speed Values

In [22]:
combined_df = combined_df[
    combined_df["speed_kmh"] <= 40
]

print("Data after removing unrealistic speeds:", combined_df.shape)

Data after removing unrealistic speeds: (7304494, 21)


### 2.7 Check Distance Values

In [23]:
combined_df["distance_km"].describe()

count    7.304494e+06
mean     1.817763e+00
std      1.546927e+00
min      0.000000e+00
25%      8.072911e-01
50%      1.363385e+00
75%      2.321769e+00
max      2.396003e+01
Name: distance_km, dtype: float64

### 2.8 Check Unrealistic Distance Values

In [24]:
# Check zero-distance trips
print(
    "Zero-distance trips:",
    (combined_df["distance_km"] == 0).sum()
)

# Check trips longer than 25 km
print(
    "Trips longer than 25 km:",
    (combined_df["distance_km"] > 25).sum()
)

# Display distance statistics
print("\nDistance statistics:")
print(combined_df["distance_km"].describe())

Zero-distance trips: 112851
Trips longer than 25 km: 0

Distance statistics:
count    7.304494e+06
mean     1.817763e+00
std      1.546927e+00
min      0.000000e+00
25%      8.072911e-01
50%      1.363385e+00
75%      2.321769e+00
max      2.396003e+01
Name: distance_km, dtype: float64


In [25]:
# Select zero-distance trips
zero_distance_trips = combined_df[
    combined_df["distance_km"] == 0
]

# Display sample records
zero_distance_trips[
    [
        "start_station_name",
        "end_station_name",
        "trip_duration",
        "distance_km"
    ]
].head(10)

,start_station_name,end_station_name,trip_duration,distance_km
149,Mott St & Prince St,Mott St & Prince St,12.481067,0.0
151,Franklin St & Dupont St,Franklin St & Dupont St,21.434667,0.0
468,Ave C & E 16 St,Ave C & E 16 St,17.194717,0.0
470,Clermont Ave & Park Ave,Clermont Ave & Park Ave,1.548117,0.0
471,Ave C & E 16 St,Ave C & E 16 St,1.561267,0.0
473,Coffey St & Ferris St,Coffey St & Ferris St,1.595817,0.0
474,E 115 St & Madison Ave,E 115 St & Madison Ave,1.257817,0.0
475,W Broadway & Spring St,W Broadway & Spring St,20.020817,0.0
476,E 14 St & 1 Ave,E 14 St & 1 Ave,5.516650,0.0
869,E 11 St & Ave B,E 11 St & Ave B,6.112467,0.0


### 2.9 Final Distance Validation

In [26]:
# Check final distance validity
print("Distance validation completed.")

print(
    "Trips with distance greater than 25 km:",
    (combined_df["distance_km"] > 25).sum()
)

print(
    "Zero-distance trips retained:",
    (combined_df["distance_km"] == 0).sum()
    
)

Distance validation completed.
Trips with distance greater than 25 km: 0
Zero-distance trips retained: 112851


In [27]:
# Ride count by user type

user_type_counts = combined_df["member_casual"].value_counts()

print(user_type_counts)

member_casual
member    6467099
casual     837395
Name: count, dtype: int64


### 2.10 Save Cleaned and Feature-Engineered Dataset

In [28]:
# Save the cleaned and feature-engineered dataset

processed_path = "../data/processed/citibike_cleaned_features.csv"

combined_df.to_csv(processed_path, index=False)

print("Cleaned dataset saved successfully!")
print("Shape:", combined_df.shape)
print("Saved to:", processed_path)

Cleaned dataset saved successfully!
Shape: (7304494, 21)
Saved to: ../data/processed/citibike_cleaned_features.csv


## 3. Data Validation and Quality Checks

In [29]:
print("Total rows:", len(combined_df))

print(
    "Zero-distance trips:",
    (combined_df["distance_km"] == 0).sum()
)

print(
    "Trips above 25 km:",
    (combined_df["distance_km"] > 25).sum()
)

print(
    "Trips above 40 km/h:",
    (combined_df["speed_kmh"] > 40).sum()
)

Total rows: 7304494
Zero-distance trips: 112851
Trips above 25 km: 0
Trips above 40 km/h: 0


In [30]:
zero_distance_percentage = (
    (combined_df["distance_km"] == 0).sum()
    / len(combined_df)
) * 100

print(
    "Zero-distance percentage:",
    round(zero_distance_percentage, 2),
    "%"
)

Zero-distance percentage: 1.54 %


In [31]:
zero_distance_trips = combined_df[
    combined_df["distance_km"] == 0
]

print(
    "Average duration of zero-distance trips:",
    round(zero_distance_trips["trip_duration"].mean(), 2),
    "minutes"
)

print(
    "Maximum duration of zero-distance trips:",
    round(zero_distance_trips["trip_duration"].max(), 2),
    "minutes"
)

Average duration of zero-distance trips: 18.57 minutes
Maximum duration of zero-distance trips: 1439.0 minutes


In [32]:
long_zero_distance = zero_distance_trips[
    zero_distance_trips["trip_duration"] > 120
]

print(
    "Zero-distance trips above 2 hours:",
    len(long_zero_distance)
)

print(
    long_zero_distance["trip_duration"].describe()
)

Zero-distance trips above 2 hours: 656
count     656.000000
mean      313.562723
std       312.430354
min       120.250633
25%       135.570200
50%       174.559775
75%       312.026675
max      1438.997567
Name: trip_duration, dtype: float64


In [33]:
long_zero_distance[
    ["trip_duration", "distance_km", "start_station_name", "end_station_name"]
].head(10)

,trip_duration,distance_km,start_station_name,end_station_name
27515,898.100800,0.0,St Johns Pl & Washington Ave,St Johns Pl & Washington Ave
60888,130.253850,0.0,Adam Clayton Powell Blvd & W 126 St,Adam Clayton Powell Blvd & W 126 St
65038,151.051267,0.0,Greenwich St & Hubert St,Greenwich St & Hubert St
71243,218.393000,0.0,Frederick Douglass Blvd & W 139 St,Frederick Douglass Blvd & W 139 St
110773,175.616417,0.0,Willis Ave & Bruckner Blvd,Willis Ave & Bruckner Blvd
127859,295.149333,0.0,Westchester Ave & Jackson Ave,Westchester Ave & Jackson Ave
149998,506.690683,0.0,W 21 St & 6 Ave,W 21 St & 6 Ave
163442,281.262017,0.0,11 Ave & W 27 St,11 Ave & W 27 St
167038,252.202067,0.0,11 Ave & W 27 St,11 Ave & W 27 St
192186,141.962967,0.0,Clinton St & Grand St,Clinton St & Grand St


In [34]:
same_station_zero_distance = zero_distance_trips[
    zero_distance_trips["start_station_name"]
    == zero_distance_trips["end_station_name"]
]

print(
    "Zero-distance trips with same start and end station:",
    len(same_station_zero_distance)
)

print(
    "Percentage:",
    round(
        len(same_station_zero_distance)
        / len(zero_distance_trips) * 100,
        2
    ),
    "%"
)

Zero-distance trips with same start and end station: 112851
Percentage: 100.0 %


### Data Validation Findings

- Total trips analyzed: 7,304,494
- Zero-distance trips: 112,851 (1.54%)
- All zero-distance trips had the same start and end station.
- 656 zero-distance trips lasted more than 2 hours.
- Maximum duration among zero-distance trips: approximately 24 hours.
- No trips exceeded 25 km distance or 40 km/h speed.

Zero-distance trips were retained for transparency and further analysis.
Their potential impact should be considered when interpreting distance- and speed-related insights.

## Cleaning Decisions

- Invalid and zero-duration trips were removed.
- Trips longer than 24 hours were removed.
- Missing station information was handled during cleaning.
- Station coordinates were validated.
- Speeds above 40 km/h were removed.
- Zero-distance trips were retained and documented.
- The cleaned dataset was saved for analysis.